# 06 - Evaluation: Models Evaluation


In this notebook, we perform an analysis of all trained models using:
- Metric distribution analysis (plots and summary statistics)

We do not perform comparison among models, we just evaluate each single model as it is.


## Import libraries and set the paths

In [1]:
from __future__ import annotations

import math

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from fraud_dynamic_ensemble.config import MODELS_DIR, FIGURES_DIR

2026-01-24 16:39:10.962 | INFO     | fraud_dynamic_ensemble.config:<module>:14 - PROJ_ROOT path is: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection


In [2]:
EXPERIMENT_NAME = "CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5"

In [3]:
models_results_path: Path = MODELS_DIR / EXPERIMENT_NAME
print(f"Loading results at path:\n\t{models_results_path}")

Loading results at path:
	/home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/models/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5


In [4]:
FIGURES_MODELS_COMPARISON_DIR = FIGURES_DIR / "EV_models_comparison_evaluation"
FIGURES_MODELS_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR = FIGURES_MODELS_COMPARISON_DIR / EXPERIMENT_NAME
FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_SINGLE_MODEL_EVALUATION_DIR = FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR / "single_models_evaluation"
FIGURES_SINGLE_MODEL_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
pd.set_option("display.max_columns", None)
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
sns.set_palette("tab10")

## Data Loading and Basic Overview

In [6]:
files = list(models_results_path.glob("*/resubstitution_metrics_summary.csv"))
resubstitution_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    resubstitution_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", resubstitution_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

Found 23 files. Loading...
Success! Combined dataframe shape: (2300, 30)


In [7]:
resubstitution_df

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,fold_size,cv_tuning_mean_train_score,cv_tuning_std_train_score,cv_tuning_mean_val_score,cv_tuning_std_val_score,best_params,tuning_time,selected_features_indices,selected_features_names
0,CostSensitiveLearning_with_RandomUnderSampler_...,1,1,KNOP,resubstitution,283,17681,26,58,0.995346,0.915858,0.829912,0.870769,0.998532,0.001468,0.914222,0.910326,0.869499,0.868405,0.966930,0.856345,22560,0.867981,0.006202,0.867488,0.023827,{'classifier__estimator__ccp_alpha': np.float6...,15.475948,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
1,CostSensitiveLearning_with_RandomUnderSampler_...,1,2,KNOP,resubstitution,284,17674,33,57,0.995013,0.895899,0.832845,0.863222,0.998136,0.001864,0.915490,0.911752,0.861280,0.860686,0.963983,0.856529,22560,0.877954,0.012495,0.872459,0.020775,{'classifier__estimator__ccp_alpha': np.float6...,12.372438,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2,CostSensitiveLearning_with_RandomUnderSampler_...,1,3,KNOP,resubstitution,284,17677,30,57,0.995180,0.904459,0.832845,0.867176,0.998306,0.001694,0.915575,0.911830,0.865488,0.864725,0.963626,0.855357,22560,0.871822,0.008384,0.871632,0.032190,{'classifier__estimator__ccp_alpha': np.float6...,13.482401,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
3,CostSensitiveLearning_with_RandomUnderSampler_...,1,4,KNOP,resubstitution,280,17669,38,61,0.994515,0.880503,0.821114,0.849772,0.997854,0.002146,0.909484,0.905181,0.847518,0.846982,0.964726,0.860469,22560,0.858742,0.012604,0.863774,0.018838,{'classifier__estimator__ccp_alpha': np.float6...,12.168306,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
4,CostSensitiveLearning_with_RandomUnderSampler_...,1,5,KNOP,resubstitution,283,17684,24,57,0.995512,0.921824,0.832353,0.874807,0.998645,0.001355,0.915499,0.911715,0.873707,0.872528,0.964074,0.851361,22560,0.868887,0.006417,0.863542,0.016885,{'classifier__estimator__ccp_alpha': np.float6...,15.394208,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2295,CostSensitiveLearning_with_RandomUnderSampler_...,10,6,APriori,resubstitution,283,17679,29,57,0.995235,0.907051,0.832353,0.868098,0.998362,0.001638,0.915358,0.911586,0.866505,0.865676,0.962238,0.849020,22560,0.869964,0.002995,0.864409,0.016690,{'classifier__estimator__ccp_alpha': np.float6...,35.397440,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2296,CostSensitiveLearning_with_RandomUnderSampler_...,10,7,APriori,resubstitution,283,17685,23,57,0.995567,0.924837,0.832353,0.876161,0.998701,0.001299,0.915527,0.911741,0.875168,0.873911,0.965637,0.860144,22560,0.875174,0.010441,0.880964,0.021879,{'classifier__estimator__ccp_alpha': np.float6...,41.156827,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2297,CostSensitiveLearning_with_RandomUnderSampler_...,10,8,APriori,resubstitution,293,17678,29,48,0.995734,0.909938,0.859238,0.883861,0.998362,0.001638,0.928800,0.926191,0.882066,0.881690,0.964897,0.873448,22561,0.890696,0.010273,0.893376,0.018085,{'classifier__estimator__ccp_alpha': np.float6...,39.220807,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2298,CostSensitiveLearning_with_RandomUnderSampler_...,10,9,APriori,resubstitution,287,17685,22,54,0.995789,0.928803,0.841642,0.883077,0.998758,0.001242,0.920200,0.916841,0.882047,0.880938,0.972749,0.880140,22561,0.871643,0.013329,0.879138,0.025824,{'

In [8]:
files = list(models_results_path.glob("*/generalization_metrics_summary.csv"))
generalization_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    generalization_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", generalization_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

Found 23 files. Loading...
Success! Combined dataframe shape: (2300, 25)


In [9]:
generalization_df

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,score_time,fold_size,selected_features_indices,selected_features_names
0,CostSensitiveLearning_with_RandomUnderSampler_...,1,1,KNOP,generalization,44,2421,39,3,0.983247,0.530120,0.936170,0.676923,0.984146,0.015854,0.960158,0.959859,0.697667,0.668999,0.970611,0.944273,8.688160,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
1,CostSensitiveLearning_with_RandomUnderSampler_...,1,2,KNOP,generalization,38,2436,24,9,0.986837,0.612903,0.808511,0.697248,0.990244,0.009756,0.899377,0.894775,0.697581,0.690650,0.941502,0.826966,7.292069,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2,CostSensitiveLearning_with_RandomUnderSampler_...,1,3,KNOP,generalization,39,2432,28,8,0.985640,0.582090,0.829787,0.684211,0.988618,0.011382,0.909203,0.905728,0.688260,0.677095,0.956746,0.851520,6.796860,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
3,CostSensitiveLearning_with_RandomUnderSampler_...,1,4,KNOP,generalization,39,2441,19,8,0.989230,0.672414,0.829787,0.742857,0.992276,0.007724,0.911032,0.907402,0.741675,0.737419,0.934177,0.805781,3.963220,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
4,CostSensitiveLearning_with_RandomUnderSampler_...,1,5,KNOP,generalization,41,2379,80,7,0.965297,0.338843,0.854167,0.485207,0.967466,0.032534,0.910817,0.909053,0.525351,0.470695,0.932391,0.845433,13.709857,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2295,CostSensitiveLearning_with_RandomUnderSampler_...,10,6,APriori,generalization,43,2400,59,5,0.974471,0.421569,0.895833,0.573333,0.976007,0.023993,0.935920,0.935061,0.604754,0.561926,0.945676,0.805650,5.055284,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2296,CostSensitiveLearning_with_RandomUnderSampler_...,10,7,APriori,generalization,39,2416,43,9,0.979258,0.475610,0.812500,0.600000,0.982513,0.017487,0.897507,0.893472,0.612508,0.590099,0.969449,0.752615,7.131830,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2297,CostSensitiveLearning_with_RandomUnderSampler_...,10,8,APriori,generalization,38,2363,96,9,0.958101,0.283582,0.808511,0.419890,0.960960,0.039040,0.884735,0.881446,0.463992,0.403320,0.882196,0.622503,8.415088,2506,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2298,CostSensitiveLearning_with_RandomUnderSampler_...,10,9,APriori,generalization,38,2398,61,9,0.972067,0.383838,0.808511,0.520548,0.975193,0.024807,0.891852,0.887949,0.545789,0.508035,0.885951,0.732713,6.866291,2506,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."


In [10]:
df_results = pd.concat([resubstitution_df, generalization_df], ignore_index=True)
df_results.head(10)

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,fold_size,cv_tuning_mean_train_score,cv_tuning_std_train_score,cv_tuning_mean_val_score,cv_tuning_std_val_score,best_params,tuning_time,selected_features_indices,selected_features_names,score_time
0,CostSensitiveLearning_with_RandomUnderSampler_...,1,1,KNOP,resubstitution,283,17681,26,58,0.995346,0.915858,0.829912,0.870769,0.998532,0.001468,0.914222,0.910326,0.869499,0.868405,0.966930,0.856345,22560,0.867981,0.006202,0.867488,0.023827,{'classifier__estimator__ccp_alpha': np.float6...,15.475948,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",NaN
1,CostSensitiveLearning_with_RandomUnderSampler_...,1,2,KNOP,resubstitution,284,17674,33,57,0.995013,0.895899,0.832845,0.863222,0.998136,0.001864,0.915490,0.911752,0.861280,0.860686,0.963983,0.856529,22560,0.877954,0.012495,0.872459,0.020775,{'classifier__estimator__ccp_alpha': np.float6...,12.372438,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",NaN
2,CostSensitiveLearning_with_RandomUnderSampler_...,1,3,KNOP,resubstitution,284,17677,30,57,0.995180,0.904459,0.832845,0.867176,0.998306,0.001694,0.915575,0.911830,0.865488,0.864725,0.963626,0.855357,22560,0.871822,0.008384,0.871632,0.032190,{'classifier__estimator__ccp_alpha': np.float6...,13.482401,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
3,CostSensitiveLearning_with_RandomUnderSampler_...,1,4,KNOP,resubstitution,280,17669,38,61,0.994515,0.880503,0.821114,0.849772,0.997854,0.002146,0.909484,0.905181,0.847518,0.846982,0.964726,0.860469,22560,0.858742,0.012604,0.863774,0.018838,{'classifier__estimator__ccp_alpha': np.float6...,12.168306,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
4,CostSensitiveLearning_with_RandomUnderSampler_...,1,5,KNOP,resubstitution,283,17684,24,57,0.995512,0.921824,0.832353,0.874807,0.998645,0.001355,0.915499,0.911715,0.873707,0.872528,0.964074,0.851361,22560,0.868887,0.006417,0.863542,0.016885,{'classifier__estimator__ccp_alpha': np.float6...,15.394208,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",NaN
5,CostSensitiveLearning_with_RandomUnderSampler_...,1,6,KNOP,resubstitution,290,17689,19,50,0.996177,0.938511,0.852941,0.893683,0.998927,0.001073,0.925934,0.923053,0.892797,0.891741,0.965397,0.873432,22560,0.893335,0.009672,0.894847,0.037501,{'classifier__estimator__ccp_alpha': np.float6...,10.441921,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
6,CostSensitiveLearning_with_RandomUnderSampler_...,1,7,KNOP,resubstitution,281,17680,28,59,0.995180,0.909385,0.826471,0.865948,0.998419,0.001581,0.912445,0.908385,0.864522,0.863499,0.961075,0.847139,22560,0.874855,0.008944,0.876181,0.041145,{'classifier__estimator__ccp_alpha': np.float6...,13.625499,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
7,CostSensitiveLearning_with_RandomUnderSampler_...,1,8,KNOP,resubstitution,291,17672,35,50,0.995290,0.892638,0.853372,0.872564,0.998023,0.001977,0.925698,0.922868,0.870394,0.870166,0.977555,0.873693,22561,0.883833,0.009333,0.881443,0.016986,{'classifier__estimator__ccp_alpha': np.float6...,14.535871,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
8,CostSensitiveLearning_with_RandomUnderSampler_...,1,9,KNOP,resubstitution,280,17675,32,61,0.994847,0.897436,0.821114,0.857580,0.998193,0.001807,0.909654,0.905334,0.855837,0.854962,0.961099,0.850974,22561,0.869799,0.006538,0.867093,0.019749,{'classifier__estimator__ccp_alpha': np.float6...,11.595134,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."

In [11]:
df_results.tail(10)

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,fold_size,cv_tuning_mean_train_score,cv_tuning_std_train_score,cv_tuning_mean_val_score,cv_tuning_std_val_score,best_params,tuning_time,selected_features_indices,selected_features_names,score_time
4590,CostSensitiveLearning_with_RandomUnderSampler_...,10,1,APriori,generalization,39,2393,67,8,0.970084,0.367925,0.829787,0.509804,0.972764,0.027236,0.901276,0.898436,0.540929,0.496731,0.942367,0.674751,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",11.214456
4591,CostSensitiveLearning_with_RandomUnderSampler_...,10,2,APriori,generalization,44,2369,91,3,0.962505,0.325926,0.936170,0.483516,0.963008,0.036992,0.949589,0.949494,0.540304,0.468741,0.982287,0.847431,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",10.610579
4592,CostSensitiveLearning_with_RandomUnderSampler_...,10,3,APriori,generalization,41,2403,57,6,0.974870,0.418367,0.872340,0.565517,0.976829,0.023171,0.924585,0.923108,0.594264,0.554221,0.924122,0.770105,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",6.910542
4593,CostSensitiveLearning_with_RandomUnderSampler_...,10,4,APriori,generalization,39,2334,126,8,0.946550,0.236364,0.829787,0.367925,0.948780,0.051220,0.889284,0.887291,0.425870,0.348925,0.940218,0.780284,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",11.297779
4594,CostSensitiveLearning_with_RandomUnderSampler_...,10,5,APriori,generalization,38,2360,99,10,0.956522,0.277372,0.791667,0.410811,0.959740,0.040260,0.875703,0.871662,0.453044,0.393615,0.940304,0.691163,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",14.983176
4595,CostSensitiveLearning_with_RandomUnderSampler_...,10,6,APriori,generalization,43,2400,59,5,0.974471,0.421569,0.895833,0.573333,0.976007,0.023993,0.935920,0.935061,0.604754,0.561926,0.945676,0.805650,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",5.055284
4596,CostSensitiveLearning_with_RandomUnderSampler_...,10,7,APriori,generalization,39,2416,43,9,0.979258,0.475610,0.812500,0.600000,0.982513,0.017487,0.897507,0.893472,0.612508,0.590099,0.969449,0.752615,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",7.131830
4597,CostSensitiveLearning_with_RandomUnderSampler_...,10,8,APriori,generalization,38,2363,96,9,0.958101,0.283582,0.808511,0.419890,0.960960,0.039040,0.884735,0.881446,0.463992,0.403320,0.882196,0.622503,2506,NaN,NaN,NaN,NaN,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",8.415088
4598,CostSensitiveLearning_with_RandomUnderSampler_...,10,9,APriori,generalization,38,2398,61,9,0.972067,0.383838,0.808511,0.520548,0.975193,0.024807,0.891852,0.887949,0.545789,0.508035,0.885951,0.732713,2506,NaN,NaN,NaN,NaN,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",6.866291
4599,CostSensitiveLearning_with_RandomUnderSampler_...,10,10,APriori,generalization,41,2392,67,6,0.970870,0.379630,0.872340,0.529032,0.972753,0.027247,0.922547,0.921180,0.564542,0.516393,0.934332,0.784793,2506,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",7.333166


## Fix the metrics and general settings

In [12]:
# Define your metrics
metrics_to_analyze = [
    "balanced_accuracy",
    "mcc",
    "average_precision",
    "f1",
]
print(f"Selected metrics:\n\t{metrics_to_analyze}")

# Get list of unique models
unique_models = df_results["model"].unique()
print(f"Selected models:\n\t{unique_models}")


Selected metrics:
	['balanced_accuracy', 'mcc', 'average_precision', 'f1']
Selected models:
	['KNOP' 'KNORAE' 'APosteriori' 'KNeighborsClassifier' 'Exponential'
 'LogitBoostClassifier' 'BalancedRandomForestClassifier'
 'DecisionTreeClassifier' 'StackingClassifier' 'DESKL' 'XGBClassifier'
 'Logarithmic' 'RUSBoostClassifier' 'MLA' 'RandomForestClassifier'
 'KNORAU' 'MLPClassifier' 'ExtraTreesClassifier' 'VotingClassifier' 'RRC'
 'METADES' 'DESP' 'APriori']


## Overfitting & Generalization Gap Assessment
### Objective
To rigorously assess the model's ability to generalize to unseen data by quantifying the performance drop between the training phase (Resubstitution) and the testing phase (Generalization).

### Methodology: The Generalization Gap
For each of the 100 data partitions (10 Iterations × 10 Folds), we calculate the **Generalization Gap** ($\delta$) for every metric:

$$\delta_{metric} = \text{Score}_{train} - \text{Score}_{test}$$

* **$\delta \approx 0$:** Indicates a robust model that learns generalizable patterns (Ideal).
* **$\delta > 0$:** Indicates **Overfitting**. The model is memorizing the training data and failing to perform as well on new data.
* **$\delta < 0$:** Indicates **Underfitting** or a non-representative easy test split.

### Visualization Strategy
We utilize a hybrid **Boxplot + Strip Plot** to visualize these gaps:
1.  **Boxplot:** Displays the statistical summary (Median, IQR) of the gap distribution, providing a quick view of the "average" overfitting severity.
2.  **Strip Plot:** Overlays the raw data points (100 folds) as jittered dots. This reveals the density and detects specific "outlier folds" where the model may have failed significantly.
3.  **Reference Line:** A red dashed line at $y=0$ serves as the baseline for perfect generalization.

### Interpretation Guide
* **Tight Cluster at 0:** The model is stable and trustworthy.
* **Large Positive Spread:** The model is highly sensitive to the specific data split and likely over-parameterized.
* **High Outliers:** Individual dots floating high above the boxplot indicate specific partitions where the model failed to generalize, suggesting potential data quality issues in those specific folds.

In [13]:
def analyze_generalization_gap(df, model_name, metrics_list, save_path):
    """
    Calculate and visualize the generalization gap (train minus test) for a given model.

    This helper quantifies potential overfitting by computing, for each metric in
    ``metrics_list``, the difference:

        gap = resubstitution - generalization

    using per-(iteration, fold) paired values. The function then visualizes the gap
    distribution across metrics using a boxplot (summary statistics) overlaid with a
    strip plot (per-split points). It also writes:
    - a PNG figure to ``save_path``, and
    - a CSV of summary statistics (mean, std, min, max) per metric.

    Parameters
    ----------
    df : pandas.DataFrame
        Input table containing at least the columns:
        ``"model"``, ``"iteration"``, ``"fold"``, and ``"split"``. For each metric in
        ``metrics_list``, the DataFrame must contain a corresponding numeric column.
        The ``"split"`` column is expected to include the labels ``"resubstitution"``
        and ``"generalization"`` to enable paired subtraction.
    model_name : str
        Model identifier used to filter ``df`` via ``df["model"] == model_name``.
        The same value is used to build output filenames.
    metrics_list : Sequence[str]
        List of metric column names to include in the analysis (e.g.,
        ``["f1", "average_precision", "roc_auc"]``).
    save_path : pathlib.Path
        Output directory where the plot and CSV will be saved. This function assumes
        the directory exists.

    Returns
    -------
    summary_stats : pandas.DataFrame
        Per-metric summary statistics of the computed gaps with columns
        ``["mean", "std", "min", "max"]`` and one row per metric. If no data are found
        for ``model_name``, the function returns ``None`` (early exit).

    Notes
    -----
    - The gap is computed only for (iteration, fold) pairs where both split values are
      available after pivoting. Missing split entries will propagate as missing values
      in the subtraction.
    - Interpretation: positive gaps indicate better training performance than test
      performance (a common symptom of overfitting for that metric).
    - The plot is saved as ``<model_name>_gap_analysis.png`` and the statistics as
      ``<model_name>_gap_statistics.csv`` under ``save_path``.

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["M1", "M1", "M1", "M1"],
    ...         "iteration": [1, 1, 1, 1],
    ...         "fold": [1, 1, 2, 2],
    ...         "split": ["resubstitution", "generalization", "resubstitution", "generalization"],
    ...         "f1": [0.90, 0.80, 0.88, 0.86],
    ...         "roc_auc": [0.95, 0.92, 0.94, 0.93],
    ...     }
    ... )
    >>> out = analyze_generalization_gap(df, "M1", ["f1", "roc_auc"], Path("."))
    >>> (out.loc["f1", "mean"] >= 0.0) and (out.shape[1] == 4)
    True
    """

    # 1. Filter Data for specific model
    model_data = df[df["model"] == model_name].copy()

    if model_data.empty:
        print(f"Skipping {model_name}: No data found.")
        return

    # 2. Pivot Data to align Train/Test for calculating the difference
    # We create a table where we can subtract 'generalization' from 'resubstitution' directly
    pivot_df = model_data.pivot_table(
        index=["iteration", "fold"],
        columns="split",
        values=metrics_list
    )

    # 3. Calculate the Gap
    # Gap = Resubstitution (Train) - Generalization (Test)
    gap_data = pd.DataFrame(index=pivot_df.index)
    for metric in metrics_list:
        gap_data[metric] = pivot_df[metric]["resubstitution"] - pivot_df[metric]["generalization"]

    # 4. Melt for Plotting
    gap_long = gap_data.melt(var_name="Metric", value_name="Generalization Gap")

    # 5. Plotting
    plt.figure(figsize=(14, 8))
    sns.set_style("whitegrid")

    # A. Boxplot for Statistics (White box, black lines)
    ax = sns.boxplot(
        data=gap_long,
        x="Metric",
        y="Generalization Gap",
        color="white",
        linecolor="#333333",
        width=0.6,
        fliersize=0,        # Hide outliers here (we show them in the strip plot)
        linewidth=1.5
    )

    # B. Strip Plot for Density & Outliers (Blue dots)
    sns.stripplot(
        data=gap_long,
        x="Metric",
        y="Generalization Gap",
        color="#1f77b4",
        alpha=0.4,          # Transparency allows seeing overlapping points
        jitter=0.25,        # Spreads dots horizontally
        size=4,
        ax=ax
    )

    # C. Reference Line (Zero Gap)
    plt.axhline(0, color="#d62728", linestyle="--", linewidth=2, alpha=0.8, label="Ideal Generalization (Gap=0)")

    # Styling
    plt.title(f"Generalization Gap Analysis: {model_name}\n(Positive Values = Overfitting)",
              fontsize=16, fontweight="bold", pad=20)
    plt.ylabel("Performance Drop (Train Score - Test Score)", fontweight="bold")
    plt.xlabel("Metric", fontweight="bold")
    plt.grid(True, axis="y", alpha=0.5, linestyle="--")
    plt.legend(loc="upper right", frameon=True)

    # 6. Save Plot
    fig_filename = f"{model_name}_gap_analysis.png"
    plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
    plt.close()

    # 7. Save Summary Statistics CSV
    # Transpose describe() so metrics are rows, easier to read
    summary_stats = gap_data.describe().T[["mean", "std", "min", "max"]]
    summary_csv_name = f"{model_name}_gap_statistics.csv"
    summary_stats.to_csv(save_path / summary_csv_name)

    print(f"Gap Analysis for {model_name} completed.")
    print(f" -> Plot saved to: {save_path / fig_filename}")
    print(f" -> Stats saved to: {save_path / summary_csv_name}\n")

    return summary_stats

In [14]:
for model in unique_models:
    model_name_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_name_path.mkdir(parents=True, exist_ok=True)

    analyze_generalization_gap(df_results, model, metrics_to_analyze, model_name_path)

Gap Analysis for KNOP completed.
 -> Plot saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_gap_analysis.png
 -> Stats saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_gap_statistics.csv

Gap Analysis for KNORAE completed.
 -> Plot saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNORAE/

## ROC Space Stability (The "Cloud" Check)

### Objective
To assess the reliability and consistency of the model by visualizing its "operating point" stability on unseen data (Generalization split). Unlike scalar metrics (e.g., Accuracy), this analysis reveals how the model balances Sensitivity vs. Specificity across different data partitions.

### Methodology
For each of the 100 data partitions (10 Iterations $\times$ 10 Folds), we map the model's performance into the 2D ROC Space:
* **X-Axis:** False Positive Rate ($\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}}$) or $1 - \text{Specificity}$.
* **Y-Axis:** True Positive Rate ($\text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}}$) or $\text{Recall}$.

### Visualization Strategy
1.  **Scatter Cloud:** We plot 100 distinct points (blue dots). A tight cloud indicates a stable model, while a dispersed cloud indicates high variance.
2.  **Centroid ($\star$):** A large red star represents the average operating point of the model across all folds.
3.  **Reference Lines:** * **Diagonal (Grey Dashed):** Represents a random classifier ($\text{AUC} = 0.5$).
    * **Ideal Point (Green Cross):** The top-left corner $(0, 1)$ representing perfect prediction.

### Interpretation Guide
By observing the shape and position of the "Cloud", we can diagnose specific stability issues:

* **Tight Cluster ("Bullet Hole"):**
    * *Verdict:* **Reliable.**
    * *Meaning:* The model is extremely stable. It makes the same trade-offs regardless of how the data is split.

* **Diagonal Streak:**
    * *Verdict:* **Unstable Thresholding.**
    * *Meaning:* The model is sensitive to class balance differences in specific folds, trading off Precision for Recall unpredictably.

* **Wide Dispersion ("Shotgun Blast"):**
    * *Verdict:* **High Variance.**
    * *Meaning:* The model is highly sensitive to noise. It works well on some data splits (top-left) but fails on others (bottom-right). This often suggests the need for better regularization or feature selection.

* **Points near Diagonal:**
    * *Verdict:* **Random Guessing.**
    * *Meaning:* On these specific folds, the model failed to learn any useful patterns.

In [15]:
def analyze_roc_stability(df, model_name, save_path):
    """
    Analyze ROC-space stability of a single model on the generalization split.

    This helper filters the input results table to a single ``model_name`` and the
    ``"generalization"`` split, then computes per-row ROC-space coordinates:

    - True Positive Rate (TPR): ``tp / (tp + fn)``
    - False Positive Rate (FPR): ``fp / (fp + tn)``

    A small epsilon is added to denominators to reduce the risk of division-by-zero
    in edge cases. The function visualizes the resulting (FPR, TPR) point cloud as a
    scatter plot, overlays the centroid (mean FPR/TPR) as a highlighted marker, and
    includes standard ROC-space reference elements (random-guess diagonal and the
    ideal point (0, 1)). It also exports a CSV with summary statistics, including the
    Euclidean distance from the centroid to the ideal point.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing evaluation results. Must include:
        ``"model"``, ``"split"``, ``"tp"``, ``"fp"``, ``"tn"``, and ``"fn"`` columns.
        The ``"split"`` column must contain the value ``"generalization"`` for the
        desired evaluation subset.
    model_name : str
        Model identifier used to filter rows via ``df["model"] == model_name``.
        Also used to build output filenames.
    save_path : pathlib.Path
        Directory where outputs are written. The function assumes the directory
        exists and is writable.

    Returns
    -------
    stats : pandas.DataFrame
        Single-row DataFrame summarizing ROC-space stability with columns:
        ``["Model", "Mean_TPR", "Std_TPR", "Mean_FPR", "Std_FPR", "Distance_to_Ideal"]``.
        If no matching generalization rows are found, the function returns ``None``.

    Notes
    -----
    - This analysis uses only the ``"generalization"`` split to reflect expected
      deployment behavior.
    - TPR and FPR are computed from confusion-matrix counts. If a fold contains no
      positives or no negatives, denominators can be zero; an epsilon (``1e-9``) is
      added to mitigate division-by-zero.
    - The centroid is computed as the mean of per-row TPR and FPR values.
    - Outputs:
      - Figure: ``<model_name>_roc_stability_cloud.png``
      - Statistics: ``<model_name>_roc_stability_stats.csv``

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["M1", "M1"],
    ...         "split": ["generalization", "generalization"],
    ...         "tp": [80, 75],
    ...         "fn": [20, 25],
    ...         "fp": [10, 12],
    ...         "tn": [890, 888],
    ...     }
    ... )
    >>> out = analyze_roc_stability(df, "M1", Path("."))
    >>> (out is None) or (0.0 <= float(out["Mean_TPR"].iloc[0]) <= 1.0)
    True
    """

    # 1. Filter Data: Generalization Split Only
    # We focus on the test set to evaluate true reliability
    model_data = df[(df["model"] == model_name) & (df["split"] == "generalization")].copy()

    if model_data.empty:
        print(f"Skipping {model_name}: No generalization data found.")
        return

    # 2. Calculate ROC Coordinates (TPR vs FPR) per fold
    # We add a tiny epsilon (1e-9) to the denominator to prevent DivisionByZero errors
    # in the rare edge case where a fold has 0 positives or 0 negatives.
    model_data["tpr_calc"] = model_data["tp"] / (model_data["tp"] + model_data["fn"] + 1e-9)
    model_data["fpr_calc"] = model_data["fp"] / (model_data["fp"] + model_data["tn"] + 1e-9)

    # 3. Calculate Centroid (Mean Point)
    mean_tpr = model_data["tpr_calc"].mean()
    mean_fpr = model_data["fpr_calc"].mean()

    # 4. Create Plot
    plt.figure(figsize=(10, 10)) # Square aspect ratio is standard for ROC
    sns.set_style("whitegrid")

    # Plot the "Cloud" of individual folds
    plt.scatter(
        model_data["fpr_calc"],
        model_data["tpr_calc"],
        c="#1f77b4",
        alpha=0.6,    # Transparency helps visualize density where points overlap
        s=60,
        edgecolor="white",
        label="Individual Folds (100)"
    )

    # Plot the Centroid
    plt.scatter(
        mean_fpr,
        mean_tpr,
        c="#d62728", # Red
        s=300,
        marker="*",
        edgecolor="black",
        zorder=10,
        label=f"Mean Point (TPR={mean_tpr:.2f}, FPR={mean_fpr:.2f})"
    )

    # Plot Reference Lines
    plt.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=2, label="Random Guess")
    plt.scatter(0, 1, c="green", s=100, marker="P", label="Ideal Point (0,1)")

    # Styling
    plt.title(f"Stability Analysis: {model_name}\n(ROC Space Distribution)", fontsize=16, fontweight="bold")
    plt.xlabel("False Positive Rate (1 - Specificity)", fontweight="bold")
    plt.ylabel("True Positive Rate (Sensitivity)", fontweight="bold")
    plt.xlim(-0.02, 1.02)
    plt.ylim(-0.02, 1.02)
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)

    # Legend Positioning
    plt.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=2,
        frameon=True,
        fontsize=12,
        shadow=True
    )
    plt.subplots_adjust(bottom=0.2) # Ensure space for the legend

    # 5. Save Figure
    fig_filename = f"{model_name}_roc_stability_cloud.png"
    plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
    plt.close()

    # 6. Save Statistics
    # We calculate the Euclidean distance to the ideal point (0,1) as a summary metric
    stats = pd.DataFrame({
        "Model": [model_name],
        "Mean_TPR": [mean_tpr],
        "Std_TPR": [model_data["tpr_calc"].std()],
        "Mean_FPR": [mean_fpr],
        "Std_FPR": [model_data["fpr_calc"].std()],
        "Distance_to_Ideal": [np.sqrt(mean_fpr**2 + (1-mean_tpr)**2)]
    })

    csv_filename = f"{model_name}_roc_stability_stats.csv"
    stats.to_csv(save_path / csv_filename, index=False)

    print(f"ROC Stability Analysis for {model_name} completed.")
    print(f" -> Plot saved to: {save_path / fig_filename}")
    print(f" -> Stats saved to: {save_path / csv_filename}\n")

    return stats

In [16]:
for model in unique_models:
    model_name_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_name_path.mkdir(parents=True, exist_ok=True)

    analyze_roc_stability(df_results, model, model_name_path)

ROC Stability Analysis for KNOP completed.
 -> Plot saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_roc_stability_cloud.png
 -> Stats saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_roc_stability_stats.csv

ROC Stability Analysis for KNORAE completed.
 -> Plot saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/

## Partition Stability (Heatmap Grid)

### Objective
To diagnose whether the model's performance is consistent across all data partitions or if it relies on specific "lucky" random splits. This analysis visualizes the score variability across every single Iteration and Fold.

### Methodology
We construct a **Heatmap Grid** for each metric:
* **Grid Structure:** 10 Rows (Iterations) $\times$ 10 Columns (Folds).
* **Color Intensity:** Represents the metric score (fixed range $0.0$ to $1.0$) using the **"brg"** palette.
    * **Green:** High Performance ($\approx 1.0$).
    * **Red:** Medium Performance ($\approx 0.5$).
    * **Blue:** Low Performance ($\approx 0.0$).

### Interpretation Guide
By scanning the grid patterns, we can detect three types of behavior:

1.  **Uniform Green:**
    * *Verdict:* **Robust & High Performing.**
    * *Meaning:* The model consistently achieves high scores regardless of how the data is sliced.

2.  **"TV Static" (Random Variation):**
    * *Verdict:* **Normal Variance.**
    * *Meaning:* Slight color variations (shades of green or light green) are expected, provided there are no deep red or blue patches.

3.  **Blue/Red Stripes (Rows or Columns):**
    * *Verdict:* **Suspicious Data Sensitivity.**
    * *Meaning:*
        * **Row Stripe:** A specific Iteration (Random Seed) created a partition where the Test set was consistently harder.
        * **Column Stripe:** A specific Fold number is problematic across iterations.

4.  **Single Blue Cell:**
    * *Verdict:* **"Black Swan" Event.**
    * *Meaning:* A specific combination of data broke the model (e.g., score dropped from 0.9 to 0.4). This warrants investigation into that specific fold's data distribution.

In [17]:
def analyze_partition_stability(df, model_name, metrics_list, save_path):
    """
    Visualize partition stability across iterations and folds using a grid of heatmaps.

    This function analyzes performance variability for a single ``model_name`` on the
    ``"generalization"`` split by creating one heatmap per metric in ``metrics_list``.
    For each metric, it pivots the filtered data into an (iteration × fold) matrix and
    renders a heatmap using a fixed color scale in the range ``[0.0, 1.0]`` with the
    ``"nrg"`` colormap. A shared horizontal colorbar is added at the bottom of the
    figure to ensure consistent interpretation across metrics.

    If ``metrics_list`` contains multiple metrics, subplots are arranged in a compact
    grid with two columns (and enough rows to fit all metrics). Any unused subplot axes
    are removed.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing evaluation results. Must include columns:
        ``"model"``, ``"split"``, ``"iteration"``, and ``"fold"``, plus all metric
        columns specified in ``metrics_list``. The function filters rows where
        ``df["model"] == model_name`` and ``df["split"] == "generalization"``.
    model_name : str
        Model identifier used to filter rows via ``df["model"] == model_name``.
        Also used to build the output filename.
    metrics_list : Sequence[str]
        List of metric column names to visualize. Each entry must correspond to a
        numeric column in ``df``. Heatmaps are created in the same order as provided.
    save_path : pathlib.Path
        Directory where the heatmap grid image is saved. The function assumes the
        directory exists and is writable.

    Returns
    -------
    None
        This function returns ``None``. If no matching rows are found for the selected
        model and split, it prints a message and returns early.

    Notes
    -----
    - The visualization uses a fixed intensity range (``vmin=0.0``, ``vmax=1.0``) for
      all heatmaps and the shared colorbar. Ensure the metrics plotted are naturally
      bounded in ``[0, 1]`` (e.g., AUC, F1, accuracy) for meaningful interpretation.
    - Heatmaps are annotated with values formatted to two decimal places.
    - Output file:
      - ``<model_name>_stability_heatmap_grid.png`` saved under ``save_path``.

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["M1", "M1", "M1", "M1"],
    ...         "split": ["generalization"] * 4,
    ...         "iteration": [1, 1, 2, 2],
    ...         "fold": [1, 2, 1, 2],
    ...         "roc_auc": [0.91, 0.89, 0.92, 0.90],
    ...         "f1": [0.70, 0.68, 0.72, 0.69],
    ...     }
    ... )
    >>> analyze_partition_stability(df, "M1", ["roc_auc", "f1"], Path("."))
    >>> True
    True
    """

    # 1. Filter Data: Model + Generalization Split
    mask = (df["model"] == model_name) & (df["split"] == "generalization")
    model_data = df[mask].copy()

    if model_data.empty:
        print(f"Skipping {model_name}: No generalization data found.")
        return

    # 2. Setup Subplots Grid
    num_metrics = len(metrics_list)
    cols = 2 if num_metrics > 1 else 1
    rows = math.ceil(num_metrics / cols)

    # Dynamic figure size
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))

    # Flatten axes
    if num_metrics > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    # Global Title
    fig.suptitle(f"Partition Stability Analysis: {model_name}",
                 fontsize=20, fontweight='bold', y=0.98)

    # 3. Loop through metrics
    for i, metric in enumerate(metrics_list):
        ax = axes[i]

        # Prepare Pivot Table
        heatmap_data = model_data.pivot(index="iteration", columns="fold", values=metric)

        # Draw Heatmap
        sns.heatmap(
            heatmap_data,
            ax=ax,
            annot=True,
            fmt=".2f",
            cmap="brg",
            cbar=False,
            linewidths=.5,
            vmin=0.0,
            vmax=1.0
        )

        ax.set_title(f"Metric: {metric.upper()}", fontsize=14, fontweight="bold")
        ax.set_xlabel("Fold", fontsize=14)
        ax.set_ylabel("Iteration", fontsize=14)

    # 4. Hide empty subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # 5. Add Common Horizontal Colorbar at Bottom
    plt.tight_layout()
    fig.subplots_adjust(bottom=0.12)

    # Define position: [left, bottom, width, height]
    cbar_ax = fig.add_axes([0.15, 0.06, 0.7, 0.025])

    norm = plt.Normalize(vmin=0.0, vmax=1.0)
    sm = plt.cm.ScalarMappable(cmap="brg", norm=norm)
    sm.set_array([])

    # Define ticks every 0.05
    ticks_range = np.arange(0, 1.05, 0.05)

    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal", ticks=ticks_range)
    cbar.set_label("Score Intensity (0.0 - 1.0) | Blue=Low, Green=High", fontsize=12, fontweight="bold")

    # 6. Save Figure
    fig_filename = f"{model_name}_stability_heatmap_grid.png"
    plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Heatmap Grid for {model_name} -> Saved at path:\n\t {save_path}\n")

In [18]:
for model in unique_models:
    model_name_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_name_path.mkdir(parents=True, exist_ok=True)

    analyze_partition_stability(df_results, model, metrics_to_analyze, model_name_path)

Heatmap Grid for KNOP -> Saved at path:
	 /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP

Heatmap Grid for KNORAE -> Saved at path:
	 /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNORAE

Heatmap Grid for APosteriori -> Saved at path:
	 /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning_RandomUnderSampler___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/APosteriori

Heatmap Grid for KNeighborsC